Name Entity Recognition NLP Problem (NER)

In [1]:
import pandas as pd

In [2]:
%%time
df = pd.read_csv("ner_dataset.csv", encoding="latin1")
df = df.ffill()
print(df.head(30))

     Sentence #           Word  POS    Tag
0   Sentence: 1      Thousands  NNS      O
1   Sentence: 1             of   IN      O
2   Sentence: 1  demonstrators  NNS      O
3   Sentence: 1           have  VBP      O
4   Sentence: 1        marched  VBN      O
5   Sentence: 1        through   IN      O
6   Sentence: 1         London  NNP  B-geo
7   Sentence: 1             to   TO      O
8   Sentence: 1        protest   VB      O
9   Sentence: 1            the   DT      O
10  Sentence: 1            war   NN      O
11  Sentence: 1             in   IN      O
12  Sentence: 1           Iraq  NNP  B-geo
13  Sentence: 1            and   CC      O
14  Sentence: 1         demand   VB      O
15  Sentence: 1            the   DT      O
16  Sentence: 1     withdrawal   NN      O
17  Sentence: 1             of   IN      O
18  Sentence: 1        British   JJ  B-gpe
19  Sentence: 1         troops  NNS      O
20  Sentence: 1           from   IN      O
21  Sentence: 1           that   DT      O
22  Sentenc

In [3]:
%%time
# Converting to sentence format
sentences = df.groupby("Sentence #").apply(
    lambda s: [(w, t) for w, t in zip(s["Word"], s["Tag"])]
).tolist()
sentences[0]

CPU times: total: 13.8 s
Wall time: 14.6 s


[('Thousands', 'O'),
 ('of', 'O'),
 ('demonstrators', 'O'),
 ('have', 'O'),
 ('marched', 'O'),
 ('through', 'O'),
 ('London', 'B-geo'),
 ('to', 'O'),
 ('protest', 'O'),
 ('the', 'O'),
 ('war', 'O'),
 ('in', 'O'),
 ('Iraq', 'B-geo'),
 ('and', 'O'),
 ('demand', 'O'),
 ('the', 'O'),
 ('withdrawal', 'O'),
 ('of', 'O'),
 ('British', 'B-gpe'),
 ('troops', 'O'),
 ('from', 'O'),
 ('that', 'O'),
 ('country', 'O'),
 ('.', 'O')]

In [4]:
#creating corpus for words from sentences and tags
X = [[w for w, t in s] for s in sentences]
y = [[t for w, t in s] for s in sentences]
print("X: ", len(X), "\n y: ", len(y))

X:  47959 
 y:  47959


CRF

In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Before spliting train data into taining and validation data")
print("X_train length: ", len(X_train), X_train[0])
print("y_train length: ", len(y_train), y_train[0])
print("X_test length: ", len(X_test), X_test[0])
print("y_test length: ", len(y_test), y_test[0])
      
X_train, X_validate, y_train, y_validate=train_test_split(X_train, y_train, test_size=0.1, random_state=42)
print("After spliting train data into taining and validation data")
print("X_train length: ", len(X_train), X_train[0])
print("y_train length: ", len(y_train), y_train[0])
print("X_test length: ", len(X_test), X_test[0])
print("y_test length: ", len(y_test), y_test[0])
print("X_validate length: ", len(X_validate), X_validate[0])
print("y_validate length: ", len(y_validate), y_validate[0])

Before spliting train data into taining and validation data
X_train length:  38367 ['South', 'Korea', "'s", 'government', 'Tuesday', 'also', 'unveiled', 'a', 'so-called', 'Green', 'New', 'Job', 'Creation', 'Plan', ',', 'expected', 'to', 'create', '9,60,000', 'new', 'jobs', '.']
y_train length:  38367 ['B-geo', 'I-geo', 'O', 'O', 'B-tim', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
X_test length:  9592 ['The', 'report', 'calls', 'on', 'President', 'Bush', 'and', 'Congress', 'to', 'urge', 'Chinese', 'officials', 'not', 'to', 'use', 'the', 'global', 'war', 'against', 'terrorism', 'as', 'a', 'pretext', 'to', 'suppress', 'minorities', "'", 'rights', '.']
y_test length:  9592 ['O', 'O', 'O', 'O', 'B-per', 'I-per', 'O', 'B-org', 'O', 'O', 'B-gpe', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
After spliting train data into taining and validation data
X_train length:  34530 ['U.S.', 'military', 'lawyers', 'wan

In [6]:
%%time
# Feature Engineering
def word2features(sent, i):
    word = sent[i]
    features = {
        "word.lower()": word.lower(),
        "word[-3:]": word[-3:],
        "is_upper": word.isupper(),
        "is_title": word.istitle(),
        "is_digit": word.isdigit(),
    }

    if i > 0:
        features["-1:word"] = sent[i-1]
    else:
        features["BOS"] = True

    if i < len(sent) - 1:
        features["+1:word"] = sent[i+1]
    else:
        features["EOS"] = True

    return features

def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

CPU times: total: 0 ns
Wall time: 0 ns


In [7]:
from sklearn_crfsuite import CRF
from sklearn_crfsuite.metrics import flat_classification_report

X_train_feat = [sent2features(s) for s in X_train]
X_test_feat = [sent2features(s) for s in X_test]
print("X_train_feat: ", X_train_feat[0])
print("X_test_feat: ", X_test_feat[0])

X_train_feat:  [{'word.lower()': 'u.s.', 'word[-3:]': '.S.', 'is_upper': True, 'is_title': True, 'is_digit': False, 'BOS': True, '+1:word': 'military'}, {'word.lower()': 'military', 'word[-3:]': 'ary', 'is_upper': False, 'is_title': False, 'is_digit': False, '-1:word': 'U.S.', '+1:word': 'lawyers'}, {'word.lower()': 'lawyers', 'word[-3:]': 'ers', 'is_upper': False, 'is_title': False, 'is_digit': False, '-1:word': 'military', '+1:word': 'want'}, {'word.lower()': 'want', 'word[-3:]': 'ant', 'is_upper': False, 'is_title': False, 'is_digit': False, '-1:word': 'lawyers', '+1:word': 'charges'}, {'word.lower()': 'charges', 'word[-3:]': 'ges', 'is_upper': False, 'is_title': False, 'is_digit': False, '-1:word': 'want', '+1:word': 'dropped'}, {'word.lower()': 'dropped', 'word[-3:]': 'ped', 'is_upper': False, 'is_title': False, 'is_digit': False, '-1:word': 'charges', '+1:word': 'against'}, {'word.lower()': 'against', 'word[-3:]': 'nst', 'is_upper': False, 'is_title': False, 'is_digit': False, '-

In [8]:
%%time
crf = CRF(max_iterations=100)
crf.fit(X_train_feat, y_train)

y_pred_crf = crf.predict(X_test_feat)
print(flat_classification_report(y_test, y_pred_crf))

              precision    recall  f1-score   support

       B-art       0.00      0.00      0.00        94
       B-eve       0.56      0.26      0.35        70
       B-geo       0.84      0.91      0.88      7558
       B-gpe       0.95      0.91      0.93      3142
       B-nat       0.48      0.33      0.39        40
       B-org       0.81      0.67      0.74      4151
       B-per       0.83      0.78      0.81      3400
       B-tim       0.93      0.86      0.89      4077
       I-art       0.00      0.00      0.00        84
       I-eve       0.44      0.11      0.17        65
       I-geo       0.81      0.79      0.80      1462
       I-gpe       0.94      0.52      0.67        33
       I-nat       0.00      0.00      0.00        13
       I-org       0.79      0.79      0.79      3394
       I-per       0.82      0.89      0.86      3406
       I-tim       0.81      0.79      0.80      1251
           O       0.99      0.99      0.99    177590

    accuracy              

In [10]:
#Save the CRF model as skops file
import skops.io as skio

# CRF uses custom classes, so this is required
skio.dump(crf,"crf_model.skops") #, trusted=True)

print("CRF model saved using skops!")


CRF model saved using skops!


In [18]:
#load the model using skops
import skops.io as skio


 # Load the serialized file first
with open("crf_model.skops", "rb") as f:
    file_bytes = f.read()

# Get all untrusted types in this model
unsafe = skio.get_untrusted_types(data=file_bytes)
print("Review these types:", unsafe)

# Load the model while explicitly trusting these types
crf_loaded = skio.load("crf_model.skops", trusted=unsafe)

print("CRF model loaded successfully!")


Review these types: ['pycrfsuite._logparser.TrainLogParser', 'sklearn_crfsuite._fileresource.FileResource', 'sklearn_crfsuite.estimator.CRF']
CRF model loaded successfully!


In [19]:
# Prepare test features again
X_test_feat = [sent2features(s) for s in X_test]

# Predict with loaded model
y_pred_loaded = crf_loaded.predict(X_test_feat)

In [20]:
# Evaluate using seqeval

from seqeval.metrics import classification_report, f1_score, accuracy_score

print("F1 Score:", f1_score(y_test, y_pred_loaded))
print("Accuracy:", accuracy_score(y_test, y_pred_loaded))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_loaded))

F1 Score: 0.828157537347216
Accuracy: 0.9676595339084021

Classification Report:

              precision    recall  f1-score   support

         art       0.00      0.00      0.00        94
         eve       0.56      0.26      0.35        70
         geo       0.84      0.91      0.87      7558
         gpe       0.95      0.91      0.93      3142
         nat       0.44      0.30      0.36        40
         org       0.78      0.64      0.70      4151
         per       0.77      0.72      0.75      3400
         tim       0.91      0.83      0.87      4077

   micro avg       0.85      0.81      0.83     22532
   macro avg       0.66      0.57      0.60     22532
weighted avg       0.84      0.81      0.82     22532



LSTM 

In [9]:
%%time
#Build Vocabulary
from collections import defaultdict

word2idx = {"PAD": 0, "UNK": 1}
tag2idx = {}

for sent in X_train:
    for word in sent:
        if word not in word2idx:
            word2idx[word] = len(word2idx)

for tags in y_train:
    for tag in tags:
        if tag not in tag2idx:
            tag2idx[tag] = len(tag2idx)

CPU times: total: 344 ms
Wall time: 411 ms


In [10]:
print("word2idx len: ", len(word2idx))
print("tag2idx len: ", len(tag2idx), "\ntag2idx: ", tag2idx)

word2idx len:  30596
tag2idx len:  17 
tag2idx:  {'B-geo': 0, 'O': 1, 'B-tim': 2, 'I-tim': 3, 'I-geo': 4, 'B-org': 5, 'B-gpe': 6, 'B-per': 7, 'I-per': 8, 'I-org': 9, 'B-art': 10, 'I-art': 11, 'B-eve': 12, 'I-eve': 13, 'B-nat': 14, 'I-gpe': 15, 'I-nat': 16}


In [11]:
%%time
#Encode & Pad the data to fixed length

import torch
from torch.nn.utils.rnn import pad_sequence

def encode(seq, mapping):
    return [mapping.get(x, mapping["UNK"]) for x in seq]

X_train_enc = [torch.tensor(encode(s, word2idx)) for s in X_train]
y_train_enc = [torch.tensor([tag2idx[t] for t in s]) for s in y_train]

X_train_pad = pad_sequence(X_train_enc, batch_first=True)
y_train_pad = pad_sequence(y_train_enc, batch_first=True)


CPU times: total: 12.3 s
Wall time: 12.7 s


In [12]:
%%time
#Model
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class BiLSTM_NER(nn.Module):
    def __init__(self, vocab_size, tag_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 128)
        self.lstm = nn.LSTM(128, 128, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(256, tag_size)

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.lstm(x)
        return self.fc(x)


CPU times: total: 0 ns
Wall time: 0 ns


In [13]:
class NERDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [14]:
train_dataset = NERDataset(X_train_pad, y_train_pad)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,      # LOWER BATCH SIZE TO FIX MEMORY
    shuffle=True
)

In [15]:
%%time
#Train BiLSTM model
model = BiLSTM_NER(len(word2idx), len(tag2idx))
loss_fn = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(3):
    model.train()
    total_loss = 0.0
    for X_batch, Y_batch in train_loader:
        optimizer.zero_grad()
        # Forward pass
        outputs = model(X_batch)                     # (B, L, num_tags)
        # Reshape for loss
        loss = loss_fn(
            outputs.reshape(-1, len(tag2idx)),       # (B*L, num_tags)
            Y_batch.reshape(-1)                      # (B*L)
        )
        # Backpropagation
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

Epoch 1 Loss: 489.1009
Epoch 2 Loss: 205.7303
Epoch 3 Loss: 137.9718
CPU times: total: 2h 6min 50s
Wall time: 24min 55s


In [16]:
#save the BiLSTM model as file
torch.save(model.state_dict(), "bilstm_model.pth")

In [17]:
#load the BiLSTM model for prediction
model = BiLSTM_NER(len(word2idx), len(tag2idx))
model.load_state_dict(torch.load("bilstm_model.pth"))
model.eval()

BiLSTM_NER(
  (embedding): Embedding(30596, 128)
  (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=256, out_features=17, bias=True)
)

In [18]:
# Prediction for seqeval (BiLSTM)

def predict_bilstm(model, X):
    model.eval()
    predictions = []

    with torch.no_grad():
        for sentence in X:
            logits = model(sentence.unsqueeze(0))  # (1, L, tags)
            pred_ids = torch.argmax(logits, dim=-1).squeeze(0).tolist()
            predictions.append(pred_ids)

    return predictions

In [27]:
# Convert indices → tags
idx2tag = {v: k for k, v in tag2idx.items()}

# Encode test sequences
X_test_enc = [torch.tensor(encode(s, word2idx)) for s in X_test]

# ---- FIX: Predict each sentence separately ----
preds_idx = []
for sent in X_test_enc:
    sent = sent.unsqueeze(0)
    pred = predict_bilstm(model, sent)[0]
    preds_idx.append(pred)

# ---- Convert predictions to tags ----
preds = [
    [idx2tag[p] for p in seq]
    for seq in preds_idx
]

# ---- True labels directly from y_test (no padding!) ----
trues = [
    [tag for tag in seq]
    for seq in y_test
]

In [28]:
# Evaluation using seqeval 
from seqeval.metrics import classification_report, f1_score, accuracy_score

print("Accuracy:", accuracy_score(trues, preds))
print("F1 Score:", f1_score(trues, preds))
print("\nClassification Report:\n")
print(classification_report(trues, preds))

Accuracy: 0.9308630796358958
F1 Score: 0.5267482555485512

Classification Report:

              precision    recall  f1-score   support

         art       0.00      0.00      0.00        94
         eve       0.30      0.27      0.28        70
         geo       0.27      0.07      0.11      7558
         gpe       0.67      0.94      0.78      3142
         nat       0.34      0.35      0.35        40
         org       0.33      0.71      0.45      4151
         per       0.69      0.65      0.67      3400
         tim       0.72      0.84      0.78      4077

   micro avg       0.52      0.54      0.53     22532
   macro avg       0.41      0.48      0.43     22532
weighted avg       0.48      0.54      0.47     22532

